## Forecast 2026/27 + Model comparison

Train **both** models on all data through **2025/26** (`2526`), then simulate the **2026/27** table for the confirmed 20-team squad.

Compare Model 1 (static strengths) vs Model 2 (use **2526** season parameters).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import arviz as az
from cmdstanpy import CmdStanModel

from helping_functions import (
    load_matches,
    prepare_stan_data,
    prepare_stan_data_hierarchical,
    compute_table,
    pl_2627_squad,
    FORECAST_TRAIN_SEASONS,
    simulate_seasons_from_draws,
    simulate_seasons_from_hierarchical_draws,
)

In [ ]:
matches = load_matches()
forecast_teams = pl_2627_squad(matches)

print(f"Forecast season: 2026/27 ({len(forecast_teams)} teams)")
for i, t in enumerate(forecast_teams, 1):
    print(f"  {i:2d}. {t}")

### Model 1 — fit on full history through 2526

In [ ]:
stan1, team_to_idx, _ = prepare_stan_data(matches, FORECAST_TRAIN_SEASONS)
print(f"Model 1: N={stan1['N']}, T={stan1['T']}")

fit1 = CmdStanModel(stan_file="stan/poisson.stan").sample(
    data=stan1, seed=42, chains=4, parallel_chains=4,
    iter_warmup=1000, iter_sampling=1000, show_progress=True,
)
print(fit1.diagnose())

In [ ]:
pred1 = simulate_seasons_from_draws(
    fit1, forecast_teams, team_to_idx, n_table_sims=800, seed=1
)
pred1 = pred1.rename(columns={
    "pos_median": "pos_m1", "pts_median": "pts_m1", "pos_mean": "pos_mean_m1"
})
pred1.head(10)

### Model 2 — fit hierarchical, forecast with 2526 strengths

In [ ]:
stan2, team_to_idx2, _, season_to_idx = prepare_stan_data_hierarchical(
    matches, FORECAST_TRAIN_SEASONS
)
last_season_idx = season_to_idx["2526"]
print(f"Model 2: N={stan2['N']}, S={stan2['S']}, T={stan2['T']}")
print(f"Using season index {last_season_idx} (2526) for forecast")

fit2 = CmdStanModel(stan_file="stan/hierarchical.stan").sample(
    data=stan2, seed=42, chains=4, parallel_chains=4,
    iter_warmup=1500, iter_sampling=1500, show_progress=True,
)
print(fit2.diagnose())

In [ ]:
pred2 = simulate_seasons_from_hierarchical_draws(
    fit2, forecast_teams, team_to_idx2,
    last_season_index=last_season_idx, n_table_sims=800, seed=2
)
pred2 = pred2.rename(columns={
    "pos_median": "pos_m2", "pts_median": "pts_m2", "pos_mean": "pos_mean_m2"
})
pred2.head(10)

### Side-by-side forecast 2026/27

In [ ]:
comparison = pred1[["team", "pos_m1", "pts_m1"]].merge(
    pred2[["team", "pos_m2", "pts_m2"]], on="team"
)
comparison["pos_diff"] = comparison["pos_m1"] - comparison["pos_m2"]
comparison = comparison.sort_values("pos_m1")
comparison

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(comparison["pos_m2"], comparison["pos_m1"])
for _, r in comparison.iterrows():
    ax.annotate(r["team"], (r["pos_m2"], r["pos_m1"]), fontsize=7, alpha=0.7)
ax.plot([1, 20], [1, 20], "k--", alpha=0.5)
ax.set_xlabel("Model 2 — median position")
ax.set_ylabel("Model 1 — median position")
ax.set_title("Forecast 2026/27: agreement between models")
ax.invert_xaxis()
ax.invert_yaxis()
plt.tight_layout()
plt.show()

### WAIC / LOO on training matches (same data for both models)

In [ ]:
ll1 = fit1.stan_variable("log_lik")
ll2 = fit2.stan_variable("log_lik")

loo1 = az.loo(az.from_dict(log_likelihood={"match": ll1}))
loo2 = az.loo(az.from_dict(log_likelihood={"match": ll2}))

print("Model 1 (Poisson static):")
print(loo1)
print("\nModel 2 (Hierarchical by season):")
print(loo2)
print(f"\nDelta ELPD (Model 2 - Model 1): {loo2.elpd_loo - loo1.elpd_loo:.1f} (higher is better)")

### Optional: compare to backtest on 2526 (run notebooks 02 & 03 first)

If you saved backtest MAE from **02** / **03**, note which model tracked 2025/26 better before trusting the 2627 forecast.

In [ ]:
actual_2526 = compute_table(matches, "2526")[["team", "position", "Pts"]]
actual_2526.sort_values("position").head(10)